# Credit risk: inspecting the experiment

This notebook is a retrospective walkthrough of the fixed experiment. Training lives in `risklab/train.py`; these cells do not fit or tune models. Start with the cohort, check partition integrity, then read the held-out findings.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

In [2]:
from risklab.data import TARGET, load_data, split_indices

frame = load_data()
splits, groups = split_indices(frame)
pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(idx),
            "default_rate": frame.iloc[idx][TARGET].mean(),
            "unique_profiles": len(set(groups[idx])),
        }
        for name, idx in splits.items()
    ]
)

,split,rows,default_rate,unique_profiles
0,test,5982,0.217319,5837
1,validation,2957,0.221170,2919
2,calibration,3028,0.217966,2919
3,train,18033,0.223036,17508


## Leakage check

Row IDs alone do not establish independence. Check that repeated predictor profiles are disjoint across the four partitions. The model does not use ID, demographics or the outcome as predictors.

In [3]:
from itertools import combinations

for left, right in combinations(splits, 2):
    assert set(groups[splits[left]]).isdisjoint(groups[splits[right]])
print("All six split-pair overlap checks passed.")
print("Missing source values:", int(frame.isna().sum().sum()))

All six split-pair overlap checks passed.
Missing source values: 0


## Model selection and test results

Brier score on validation selects the model. Calibration has a separate fitting split; the test table is a final comparison, not a reason to select a different model.

In [4]:
metrics = json.loads(Path("reports/metrics.json").read_text())
print("Selected:", metrics["selected_model"])
pd.DataFrame(metrics["test"]).T.round(4)

Selected: gradient_boosting


,roc_auc,average_precision,brier,log_loss
logistic,0.7587,0.5048,0.1398,0.4441
gradient_boosting,0.7817,0.5512,0.1333,0.4248
logistic_calibrated,0.7587,0.5048,0.1400,0.4444
gradient_boosting_calibrated,0.7817,0.5512,0.1333,0.4251


## Review workload

The cost units are hypothetical. Read recall alongside the fraction flagged for review: higher recall is useful only if the process can handle the additional work.

In [5]:
pd.DataFrame([metrics["operating_point"], metrics["threshold_0_5"]])[
    ["threshold", "recall", "precision", "review_rate", "false_positive_rate", "cost_per_1000"]
].round(4)

,threshold,recall,precision,review_rate,false_positive_rate,cost_per_1000
0,0.14,0.8108,0.3320,0.5308,0.4530,560.1805
1,0.50,0.3562,0.6691,0.1157,0.0489,737.8803


In [6]:
pd.read_csv("reports/subgroup_audit.csv")[
    ["attribute", "group", "n", "recall", "false_positive_rate"]
].round(4)

,attribute,group,n,recall,false_positive_rate
0,SEX,2,3637,0.7984,0.4375
1,SEX,1,2345,0.8281,0.4778
2,age_band,31_to_50,3330,0.8074,0.4245
3,age_band,under_31,2171,0.8033,0.4820
4,age_band,over_50,481,0.8595,0.5250


## What the result does not establish

There is no temporal backtest or current US population validation. Subgroup error differences warrant investigation; excluding demographics is not proof of fairness. The PSI scenario is synthetic. See `docs/VALIDATION.md` for details and `docs/INTERVIEW_NOTES.md` for questions to work through.